In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# RBFE single pair (BRD-2 vs BRD-3)

End-to-end workflow for one relative binding free energy pair:

1. Load the BRD protein and a pose for each of two ligands, register them on the data platform
2. Run system prep in RBFE mode
3. Inspect the prepared system
4. Run RBFE FEP on the prepared system

## Setup

For local mock testing, set `DO_ENV=local` in `.env`, start the mock server
(`make mock-server`), and set `JOB_WATCH_BLOCK=1` when executing via
`nbconvert --execute` (interactive Jupyter can use non-blocking
`await rbfe.watch()`).

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Pose,
    Protein,
    RBFE,
    RBFEParams,
    SystemPrep,
    PreparedSystem,
)
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

In [ ]:
from deeporigin import projects
projects.create("python-client-dev")


## 1. Load structures and register on the data platform

We use the BRD4 example protein and two congeneric ligand poses from the
bundled dataset. `sync()` and `Pose.from_sdf()` upload files (when needed) and
register records on the data platform.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.id

In [ ]:
pose1 = Pose.from_sdf(BRD_DATA_DIR / "brd-2.sdf", protein_id=protein.id)
pose2 = Pose.from_sdf(BRD_DATA_DIR / "brd-3.sdf", protein_id=protein.id)

pose1, pose2

## 2. System prep (RBFE mode)

Pass both poses to `SystemPrep` to prepare binding and solvation legs for
the pair. This runs synchronously via `deeporigin.system-prep`.

In [ ]:
try:
    systems = PreparedSystem.from_result(
        pose1_id=pose1.id,
        pose2_id=pose2.id,
        protein_id=protein.id,
    )
    system = systems[0]
except ValueError:
    system = SystemPrep(
        protein=protein,
        pose1=pose1,
        pose2=pose2,
    ).run()
system

In [ ]:
system.id

## 3. Show the prepared system

Visualize the solvated system PDB returned by system prep.

In [ ]:
system.show()

## 4. Run RBFE on the prepared system

Submit FEP only (`mode="rbfe"`) using the prepared binding/solvation XML paths.
We quote first, then confirm to start the job.

`test_run=1` shortens the simulation for exploration; use `test_run=0` for
production-quality results.

In [ ]:
rbfe = RBFE(
    prepared_systems=[system],
    params=RBFEParams(test_run=1),
)
rbfe

In [ ]:
rbfe.start(quote=True)
rbfe.estimate

In [ ]:
rbfe.confirm()

In [ ]:
task = await rbfe.watch()

## Results

In [ ]:
rbfe.get_results()

In [ ]:
rbfe.get_user_logs()